In [14]:
# !pip install faiss-cpu langchain-community

import json
from pathlib import Path
import os 
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

BASE_DIR = Path.cwd().resolve()
DATA_DIR = BASE_DIR / "data"
VECTOR_DIR = DATA_DIR / "vectorstore"
KB_FILE = DATA_DIR / "external/cultpass_rag_articles.jsonl"
API_KEY = os.getenv("VOCAREUM_API_KEY")

DATA_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

with open(KB_FILE, "r", encoding="utf-8") as f:
    records = json.load(f)

docs = []
for r in records:
    docs.append(
        Document(
            page_content=r["content"],
            metadata={
                "article_id": r["article_id"],
                "category": r["category"],
                "title": r["title"],
                "tags": ", ".join(r["tags"]),
            },
        )
    )

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://openai.vocareum.com/v1",
    api_key=API_KEY,
)

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

vectorstore.save_local(str(VECTOR_DIR))

print(f"Stored {len(docs)} documents in {VECTOR_DIR}")

Stored 25 documents in /workspace/code/project/solution/data/vectorstore
